# Import modules

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# Import Multi-task ElasticNet

In [ ]:
import joblib

model_1 = joblib.load('ElasticNet.pkl')

In [ ]:
model_features = model_1.feature_names_in_
model_features

In [ ]:
import pickle

with open('y_train_features.pkl', 'rb') as f:
    y_train_features_loaded = pickle.load(f)

print(y_train_features_loaded)

In [ ]:
metabolite_names = pd.read_csv('y_train_tissue_name_map.csv')
metabolite_names = metabolite_names.fillna('No_id')
metabolite_names

In [ ]:
hmdb = {}
for i in y_train_features_loaded:
  if metabolite_names[metabolite_names['Query'] == i]['HMDB'].size > 0:
    hmdb[i] = metabolite_names[metabolite_names['Query'] == i]['HMDB'].values[0]
  else:
    print(f"No matching 'Query' found for '{i}' in metabolite_names.")
    hmdb[i] = 'No_id'

In [ ]:
y_train_features_loaded_hmdb = []
for i in y_train_features_loaded:
 y_train_features_loaded_hmdb.append(hmdb[i])
len(y_train_features_loaded_hmdb)

In [ ]:
y_train_features_loaded = y_train_features_loaded_hmdb

# Import Random forest regressor

In [ ]:
model_2 = joblib.load('Random Forest Regressor.pkl')

# Import ensemble

In [ ]:
final_estimator = joblib.load('Ensemble.pkl')

# RNA sequencing data

In [ ]:
rna = pd.read_csv('fudan_tpm_met_genes_y_features.csv', index_col = 0)
rna

In [ ]:
rna = rna.apply(lambda x: np.log2(x + 1))
rna.head(3)

# Metabolite prediction

In [ ]:
def filter_model_features(df):
  filtered_data = df[model_features]
  return filtered_data

In [ ]:
rna_filtered = filter_model_features(rna)

In [ ]:
rna_filtered.head(2)

In [ ]:
def predict_metabolites(model, filtered_data):
  pred_metabolites = model.predict(filtered_data)
  pred_metabolites = pd.DataFrame(pred_metabolites, index = filtered_data.index, columns = y_train_features_loaded)
  return pred_metabolites

In [ ]:
#Elastic Net predictions
rna_pred_elastic = predict_metabolites(model_1, rna_filtered)

#Random forest regressor predictions
rna_pred_rf = predict_metabolites(model_2, rna_filtered)

In [ ]:
#Ensemble predictions

def predict_metabolites_ensemble(model_1, model_2, final_estimator, filtered_data):
  pred_1 = model_1.predict(filtered_data)
  pred_2 = model_2.predict(filtered_data)
  stacked_preds = np.hstack([pred_1, pred_2])
  pred_metabolites = final_estimator.predict(stacked_preds)
  pred_metabolites = pd.DataFrame(pred_metabolites, index = filtered_data.index, columns = y_train_features_loaded)

  return pred_metabolites

rna_pred_ensemble = predict_metabolites_ensemble(model_1, model_2, final_estimator, rna_filtered)

In [ ]:
rna_pred_ensemble

# LCMS data

In [ ]:
lcms = pd.read_csv('metabolites_fudan_hmdb.csv', index_col = 0)
lcms

# Model assessment

In [ ]:
from scipy.stats import spearmanr

metabolites_spearman_positive_all = {}
metabolites_spearman_negative_all = {}

def plot_spearman_per_metabolite(y_pred, y_true, model_name):
    """Plots the Spearman's correlation coefficient per metabolite."""

    spearman_coeffs = []
    for i in range(y_true.shape[1]):
        coeff, _ = spearmanr(y_true.iloc[:, i], y_pred.iloc[:, i])
        spearman_coeffs.append(coeff)

    spearman_df = pd.DataFrame({'Metabolite': y_true.columns, 'Spearman Coefficient': spearman_coeffs})

    spearman_df = spearman_df.sort_values(by=['Spearman Coefficient'], ascending=False)
    metabolites_spearman_positive = spearman_df[spearman_df['Spearman Coefficient'] > 0]
    metabolites_spearman_positive_all[model_name] = list(metabolites_spearman_positive['Metabolite'])
    metabolites_spearman_negative = spearman_df[spearman_df['Spearman Coefficient'] < 0]
    metabolites_spearman_negative_all[model_name] = list(metabolites_spearman_negative['Metabolite'])

    plt.figure(figsize=(10, 6))
    plt.title(f"Spearman's Correlation Coefficient per Metabolite ({model_name} - Fudan)")
    plt.bar(range(len(spearman_df)), spearman_df['Spearman Coefficient'])
    plt.xticks()
    plt.xlabel("Metabolite")
    plt.savefig(f'Spearmans Corr per metabolite({model_name} Fudan.svg', bbox_inches = 'tight')
    plt.show()

In [ ]:
def get_spearman_correlation(y_pred, y_true, model_name):

  correlation, p_value = spearmanr(y_true, y_pred)

  correlation_df = pd.DataFrame({'Actual': y_true.values.ravel(), 'Predicted': y_pred.values.ravel()})

  correlation, p_value = spearmanr(correlation_df['Actual'], correlation_df['Predicted'])

  correlation_df['Correlation'] = correlation
  correlation_df['P-value'] = p_value

  sns.regplot(x='Actual', y='Predicted', data=correlation_df, line_kws={'color': 'red'})
  plt.title(f'Spearman\'s Correlation ({model_name} - Fudan)')
  plt.xlabel('Actual Values')
  plt.ylabel('Predicted Values')

  plt.text(0.1, 0.9, f'Correlation: {correlation:.2f}\nP-value: {p_value:.2f}', transform=plt.gca().transAxes)
  plt.savefig(f'Spearmans Corr ({model_name} Fudan).svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
from collections import Counter


def analyse_predictions(rna_pred_elastic, rna_pred_rf, rna_pred_ensemble, lcms):

  lcms = lcms[list(set(lcms.columns).intersection(set(rna_pred_elastic.columns)))]
  rna_pred_elastic = rna_pred_elastic[list(set(lcms.columns).intersection(set(rna_pred_elastic.columns)))]
  rna_pred_rf = rna_pred_rf[list(set(lcms.columns).intersection(set(rna_pred_rf.columns)))]
  rna_pred_ensemble = rna_pred_ensemble[list(set(lcms.columns).intersection(set(rna_pred_ensemble.columns)))]

  lcms = lcms.loc[list(set(lcms.index).intersection(set(rna_pred_elastic.index)))]
  rna_pred_elastic = rna_pred_elastic.loc[list(set(lcms.index).intersection(set(rna_pred_elastic.index)))]
  rna_pred_rf = rna_pred_rf.loc[list(set(lcms.index).intersection(set(rna_pred_rf.index)))]
  rna_pred_ensemble = rna_pred_ensemble.loc[list(set(lcms.index).intersection(set(rna_pred_ensemble.index)))]

  counts = Counter(rna_pred_elastic.columns)
  duplicates = [item for item, count in counts.items() if count > 1]
  lcms = lcms.drop(columns = duplicates)
  rna_pred_elastic = rna_pred_elastic.drop(columns = duplicates)
  rna_pred_rf = rna_pred_rf.drop(columns = duplicates)
  rna_pred_ensemble = rna_pred_ensemble.drop(columns = duplicates)

  plot_spearman_per_metabolite(y_pred = rna_pred_elastic, y_true = lcms, model_name = 'Elastic Net')
  plot_spearman_per_metabolite(y_pred = rna_pred_rf, y_true = lcms, model_name = 'Random Forest Regressor')
  plot_spearman_per_metabolite(y_pred = rna_pred_ensemble, y_true = lcms, model_name = 'Ensemble')

  get_spearman_correlation(y_pred = rna_pred_elastic, y_true = lcms, model_name = 'Elastic Net')
  get_spearman_correlation(y_pred = rna_pred_rf, y_true = lcms, model_name = 'Random Forest Regressor')
  get_spearman_correlation(y_pred = rna_pred_ensemble, y_true = lcms, model_name = 'Ensemble')

In [ ]:
analyse_predictions(rna_pred_elastic = rna_pred_elastic, rna_pred_rf = rna_pred_rf, rna_pred_ensemble = rna_pred_ensemble, lcms = lcms)

In [ ]:
with open('metabolites_spearman_positive_all.pkl', 'wb') as file:
    pickle.dump(metabolites_spearman_positive_all, file)

In [ ]:
metabolites_spearman_positive_all

In [ ]:
metabolites_spearman_negative_all

In [ ]:
positive = {}
for item in metabolites_spearman_positive_all:
  positive[item] = len(metabolites_spearman_positive_all[item])
  print(item, len(metabolites_spearman_positive_all[item]))
positive = pd.DataFrame(positive, index = [0])
positive.rename(index = {0 : 'Positive'}, inplace = True)
positive

In [ ]:
negative = {}
for item in metabolites_spearman_negative_all:
  negative[item] = len(metabolites_spearman_negative_all[item])
  print(item, len(metabolites_spearman_negative_all[item]))
negative = pd.DataFrame(negative, index = [0])
negative.rename(index = {0 : 'Negative'}, inplace = True)
negative

In [ ]:
pos_neg = pd.concat([positive, negative], axis = 0)
pos_neg = pos_neg.T
pos_neg

In [ ]:
pos_neg.plot(kind = 'bar', stacked = True)
plt.legend(bbox_to_anchor = (1.05, 1), loc = 'upper left')
plt.title("Models Spearman's correlation prediction vs actual")
plt.ylabel('Counts')
plt.savefig('Fudan_model_comparison_spearmans.svg', bbox_inches = 'tight')
plt.show()

# MYC case study

In [ ]:
cnv = pd.read_csv('FUSCC_TNBC_annotation_withCNV.txt', sep = '\t')
cnv

In [ ]:
cnv.MYC_status.value_counts()

In [ ]:
myc = cnv[['Transcriptomic_Sample', 'MYC_status']]
myc = myc.dropna()
myc

In [ ]:
myc.MYC_status.isna().sum()

In [ ]:
myc_status = []
for i in myc.MYC_status:
  if i in ['MYC_Gain', 'MYC_Amplification']:
    myc_status.append('Gain/Amplification')
  else:
    myc_status.append('Loss/No_change')

In [ ]:
myc['MYC'] = myc_status
myc = myc.set_index('Transcriptomic_Sample').drop('MYC_status', axis = 'columns')
myc

In [ ]:
myc = myc.loc[myc.index.intersection(lcms.index)]
myc

In [ ]:
myc.MYC.value_counts()

In [ ]:
def select_and_add_myc_status(df, myc_df):
  common_index = list(df.index.intersection(myc_df.index))
  res = pd.concat([myc_df.loc[common_index], df.loc[common_index]], axis = 'columns')
  return res

In [ ]:
rna_pred_ensemble

In [ ]:
lcms_myc = select_and_add_myc_status(df = lcms, myc_df = myc)
elastic_pred_myc = select_and_add_myc_status(df = rna_pred_elastic, myc_df = myc)
rf_pred_myc = select_and_add_myc_status(df = rna_pred_rf, myc_df = myc)
ensemble_pred_myc = select_and_add_myc_status(df = rna_pred_ensemble, myc_df = myc)

In [ ]:
lcms_myc.head(2)

In [ ]:
def get_common_metabolites(df1, df2, df3, df4):
  common_cols = df1.columns.intersection(df2.columns)
  common_cols = common_cols.intersection(df3.columns)
  common_cols = common_cols.intersection(df4.columns)
  return list(common_cols)

common_metabolites = get_common_metabolites(lcms_myc, elastic_pred_myc, rf_pred_myc, ensemble_pred_myc)
len(common_metabolites)

In [ ]:
lcms_myc = lcms_myc[common_metabolites]
elastic_pred_myc = elastic_pred_myc[common_metabolites]
rf_pred_myc = rf_pred_myc[common_metabolites]
ensemble_pred_myc = ensemble_pred_myc[common_metabolites]

In [ ]:
def get_mean_and_direction_of_change_by_group(df):
  res = df.groupby('MYC').mean()
  res = res.transpose()
  res['Higher_in_altered'] = res['Gain/Amplification'] > res['Loss/No_change']
  res.replace({True : 'yes', False : 'no'}, inplace = True)
  return res

In [ ]:
lcms_myc_mean = get_mean_and_direction_of_change_by_group(lcms_myc)
elastic_myc_mean = get_mean_and_direction_of_change_by_group(elastic_pred_myc)
rf_myc_mean = get_mean_and_direction_of_change_by_group(rf_pred_myc)
ensemble_myc_mean = get_mean_and_direction_of_change_by_group(ensemble_pred_myc)

In [ ]:
elastic_myc_mean = elastic_myc_mean.loc[lcms_myc_mean.index][~elastic_myc_mean.index.duplicated(keep='first')]
rf_myc_mean = rf_myc_mean.loc[lcms_myc_mean.index][~rf_myc_mean.index.duplicated(keep='first')]
ensemble_myc_mean = ensemble_myc_mean.loc[lcms_myc_mean.index][~ensemble_myc_mean.index.duplicated(keep='first')]

In [ ]:
def get_pred_vs_actual_comparison(df_elastic, df_rf, df_ensemble, df_lcms, case_study_name):
  #Elastic Net
  df1 = pd.concat([df_elastic.rename(columns = {'Higher_in_altered' : 'prediction'})['prediction'], df_lcms.rename(columns = {'Higher_in_altered' : 'actual'})['actual']], axis = 'columns')
  df1['comparison'] = df1['prediction'] == df1['actual']
  print('Correctly predicted metabolites from Elastic Net: ' + str(list(df1[df1['comparison'] == True].index)))
  res1 = pd.DataFrame(df1['comparison'].value_counts())
  res1.rename(columns = {'count' : 'ElasticNet'}, inplace = True)

  #Random forest regressor
  df2 = pd.concat([df_rf.rename(columns = {'Higher_in_altered' : 'prediction'})['prediction'], df_lcms.rename(columns = {'Higher_in_altered' : 'actual'})['actual']], axis = 'columns')
  df2['comparison'] = df2['prediction'] == df2['actual']
  print('Correctly predicted metabolites from Random forest regressor: ' + str(list(df2[df2['comparison'] == True].index)))
  res2 = pd.DataFrame(df2['comparison'].value_counts())
  res2.rename(columns = {'count' : 'RF regressor'}, inplace = True)

  #Ensemble
  df3 = pd.concat([df_ensemble.rename(columns = {'Higher_in_altered' : 'prediction'})['prediction'], df_lcms.rename(columns = {'Higher_in_altered' : 'actual'})['actual']], axis = 'columns')
  df3['comparison'] = df3['prediction'] == df3['actual']
  print('Correctly predicted metabolites from Ensemble: ' + str(list(df3[df3['comparison'] == True].index)))
  res3 = pd.DataFrame(df3['comparison'].value_counts())
  res3.rename(columns = {'count' : 'Ensemble'}, inplace = True)

  #Final result
  res = pd.concat([res1, res2, res3], axis = 'columns')

  res_transposed = res.T
  res_transposed.plot(kind='bar', stacked=True)
  locs, labels = plt.yticks()

  new_labels = [int(float(label.get_text())) for label in labels]
  plt.yticks(locs, new_labels)

  plt.title(f'Comparison of Predictions vs. Actuals ({case_study_name})')
  plt.ylabel('Count')
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
  plt.savefig(f'{case_study_name}_pred_vs_actual.svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
get_pred_vs_actual_comparison(df_elastic = elastic_myc_mean,
                              df_rf = rf_myc_mean,
                              df_ensemble = ensemble_myc_mean,
                              df_lcms = lcms_myc_mean,
                              case_study_name = 'Fudan_MYC')

In [ ]:
from scipy.stats import ttest_ind

In [ ]:
def get_statistically_significant_metabolites(df, df_mean, pval_threshold):
  grouped = df.groupby('MYC')
  mut = grouped.get_group('Gain/Amplification').drop(columns=['MYC'])
  wt = grouped.get_group('Loss/No_change').drop(columns=['MYC'])
  results = {}
  for col in mut.columns:
    t_stat, p_value = ttest_ind(mut[col], wt[col])
    results[col] = {'T-statistic': t_stat, 'P-value': p_value}
  results_df = pd.DataFrame(results).T
  results_df.index = mut.columns
  df_mean = df_mean.copy()
  df_mean.index.name = None
  res = pd.concat([df_mean, results_df], axis = 'columns')
  return res

In [ ]:
myc_sig = get_statistically_significant_metabolites(df = lcms_myc, df_mean = lcms_myc_mean,  pval_threshold = 0.05)
myc_sig.head(2)
myc_sig_metabolites = list(myc_sig[myc_sig['P-value'] < 0.05].index)

In [ ]:
#For statistically siginificant metabolites
get_pred_vs_actual_comparison(df_elastic = elastic_myc_mean.loc[myc_sig_metabolites],
                              df_rf = rf_myc_mean.loc[myc_sig_metabolites],
                              df_ensemble = ensemble_myc_mean.loc[myc_sig_metabolites],
                              df_lcms = lcms_myc_mean.loc[myc_sig_metabolites],
                              case_study_name = 'Fudan_MYC_stat_sig')

# PI3K case study

In [ ]:
pi3k = cnv[['Transcriptomic_Sample', 'PI3K_activity_status']]
pi3k

In [ ]:
pi3k.PI3K_activity_status.value_counts()

In [ ]:
pi3k = pi3k[['Transcriptomic_Sample', 'PI3K_activity_status']]
pi3k = pi3k.dropna()
pi3k = pi3k.set_index('Transcriptomic_Sample')
pi3k

In [ ]:
pi3k = pi3k.loc[pi3k.index.intersection(lcms.index)]
pi3k

In [ ]:
pi3k.isna().sum()

In [ ]:
pi3k_status = []
for i in pi3k.PI3K_activity_status:
  if i  == 'PI3K_PTEN_WT':
    pi3k_status.append('WT')
  else:
    pi3k_status.append('MUT')

In [ ]:
pi3k['PI3K'] = pi3k_status
pi3k = pi3k.drop('PI3K_activity_status', axis = 'columns')
pi3k

In [ ]:
pi3k.PI3K.value_counts()

In [ ]:
def select_and_add_pi3k_status(df, pi3k_df):
  common_index = list(df.index.intersection(pi3k_df.index))
  res = pd.concat([pi3k_df.loc[common_index], df.loc[common_index]], axis = 'columns')
  return res

In [ ]:
rna_pred_ensemble

In [ ]:
lcms_pi3k = select_and_add_pi3k_status(df = lcms, pi3k_df = pi3k)
elastic_pred_pi3k = select_and_add_pi3k_status(df = rna_pred_elastic, pi3k_df = pi3k)
rf_pred_pi3k = select_and_add_pi3k_status(df = rna_pred_rf, pi3k_df = pi3k)
ensemble_pred_pi3k = select_and_add_pi3k_status(df = rna_pred_ensemble, pi3k_df = pi3k)

In [ ]:
lcms_pi3k.head(2)

In [ ]:
def get_common_metabolites(df1, df2, df3, df4):
  common_cols = df1.columns.intersection(df2.columns)
  common_cols = common_cols.intersection(df3.columns)
  common_cols = common_cols.intersection(df4.columns)
  return list(common_cols)

common_metabolites = get_common_metabolites(lcms_pi3k, elastic_pred_pi3k, rf_pred_pi3k, ensemble_pred_pi3k)
len(common_metabolites)

In [ ]:
lcms_pi3k = lcms_pi3k[common_metabolites]
elastic_pred_pi3k = elastic_pred_pi3k[common_metabolites]
rf_pred_pi3k = rf_pred_pi3k[common_metabolites]
ensemble_pred_pi3k = ensemble_pred_pi3k[common_metabolites]

In [ ]:
def get_mean_and_direction_of_change_by_group(df):
  res = df.groupby('PI3K').mean()
  res = res.transpose()
  res['Higher_in_altered'] = res['MUT'] > res['WT']
  res.replace({True : 'yes', False : 'no'}, inplace = True)
  return res

In [ ]:
lcms_pi3k_mean = get_mean_and_direction_of_change_by_group(lcms_pi3k)
elastic_pi3k_mean = get_mean_and_direction_of_change_by_group(elastic_pred_pi3k)
rf_pi3k_mean = get_mean_and_direction_of_change_by_group(rf_pred_pi3k)
ensemble_pi3k_mean = get_mean_and_direction_of_change_by_group(ensemble_pred_pi3k)

In [ ]:
elastic_pi3k_mean = elastic_pi3k_mean.loc[lcms_pi3k_mean.index][~elastic_pi3k_mean.index.duplicated(keep='first')]
rf_pi3k_mean = rf_pi3k_mean.loc[lcms_pi3k_mean.index][~rf_pi3k_mean.index.duplicated(keep='first')]
ensemble_pi3k_mean = ensemble_pi3k_mean.loc[lcms_pi3k_mean.index][~ensemble_pi3k_mean.index.duplicated(keep='first')]

In [ ]:
def get_pred_vs_actual_comparison(df_elastic, df_rf, df_ensemble, df_lcms, case_study_name):
  #Elastic Net
  df1 = pd.concat([df_elastic.rename(columns = {'Higher_in_altered' : 'prediction'})['prediction'], df_lcms.rename(columns = {'Higher_in_altered' : 'actual'})['actual']], axis = 'columns')
  df1['comparison'] = df1['prediction'] == df1['actual']
  print('Correctly predicted metabolites from Elastic Net: ' + str(list(df1[df1['comparison'] == True].index)))
  res1 = pd.DataFrame(df1['comparison'].value_counts())
  res1.rename(columns = {'count' : 'ElasticNet'}, inplace = True)

  #Random forest regressor
  df2 = pd.concat([df_rf.rename(columns = {'Higher_in_altered' : 'prediction'})['prediction'], df_lcms.rename(columns = {'Higher_in_altered' : 'actual'})['actual']], axis = 'columns')
  df2['comparison'] = df2['prediction'] == df2['actual']
  print('Correctly predicted metabolites from Random forest regressor: ' + str(list(df2[df2['comparison'] == True].index)))
  res2 = pd.DataFrame(df2['comparison'].value_counts())
  res2.rename(columns = {'count' : 'RF regressor'}, inplace = True)

  #Ensemble
  df3 = pd.concat([df_ensemble.rename(columns = {'Higher_in_altered' : 'prediction'})['prediction'], df_lcms.rename(columns = {'Higher_in_altered' : 'actual'})['actual']], axis = 'columns')
  df3['comparison'] = df3['prediction'] == df3['actual']
  print('Correctly predicted metabolites from Ensemble: ' + str(list(df3[df3['comparison'] == True].index)))
  res3 = pd.DataFrame(df3['comparison'].value_counts())
  res3.rename(columns = {'count' : 'Ensemble'}, inplace = True)

  #Final result
  res = pd.concat([res1, res2, res3], axis = 'columns')

  res_transposed = res.T
  res_transposed.plot(kind='bar', stacked=True)
  locs, labels = plt.yticks()

  new_labels = [int(float(label.get_text())) for label in labels]
  plt.yticks(locs, new_labels)

  plt.title(f'Comparison of Predictions vs. Actuals ({case_study_name})')
  plt.ylabel('Count')
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
  plt.savefig(f'{case_study_name}_pred_vs_actual.svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
get_pred_vs_actual_comparison(df_elastic = elastic_pi3k_mean,
                              df_rf = rf_pi3k_mean,
                              df_ensemble = ensemble_pi3k_mean,
                              df_lcms = lcms_pi3k_mean,
                              case_study_name = 'Fudan_PI3K')

In [ ]:
from scipy.stats import ttest_ind

In [ ]:
def get_statistically_significant_metabolites(df, df_mean, pval_threshold):
  grouped = df.groupby('PI3K')
  mut = grouped.get_group('MUT').drop(columns=['PI3K'])
  wt = grouped.get_group('WT').drop(columns=['PI3K'])
  results = {}
  for col in mut.columns:
    t_stat, p_value = ttest_ind(mut[col], wt[col])
    results[col] = {'T-statistic': t_stat, 'P-value': p_value}
  results_df = pd.DataFrame(results).T
  results_df.index = mut.columns
  df_mean = df_mean.copy()
  df_mean.index.name = None
  res = pd.concat([df_mean, results_df], axis = 'columns')
  return res

In [ ]:
pi3k_sig = get_statistically_significant_metabolites(df = lcms_pi3k, df_mean = lcms_pi3k_mean,  pval_threshold = 0.05)
pi3k_sig.head(2)
pi3k_sig_metabolites = list(pi3k_sig[pi3k_sig['P-value'] < 0.05].index)

In [ ]:
#For statistically siginificant metabolites
get_pred_vs_actual_comparison(df_elastic = elastic_pi3k_mean.loc[pi3k_sig_metabolites],
                              df_rf = rf_pi3k_mean.loc[pi3k_sig_metabolites],
                              df_ensemble = ensemble_pi3k_mean.loc[pi3k_sig_metabolites],
                              df_lcms = lcms_pi3k_mean.loc[pi3k_sig_metabolites],
                              case_study_name = 'Fudan_PI3K_stat_sig')

# Case study common metabolites

In [ ]:
myc =['HMDB0000050', 'HMDB0000220', 'HMDB0001173', 'HMDB0000162', 'HMDB0000806', 'HMDB0000517', 'HMDB0000673', 'HMDB0001406', 'HMDB0000133', 'HMDB0000641', 'HMDB0000159', 'HMDB0000848', 'HMDB0000296', 'HMDB0000251', 'HMDB0000300', 'HMDB0011737', 'HMDB0000169', 'HMDB0003337', 'HMDB0000034', 'HMDB0000148', 'HMDB0000904', 'HMDB0000684', 'HMDB0000086', 'HMDB0000638', 'HMDB0000158', 'HMDB0000827', 'HMDB0011503', 'HMDB0000929', 'HMDB0000254', 'HMDB0000043', 'HMDB0000870', 'HMDB0000625', 'HMDB0000812', 'HMDB0000759', 'HMDB0002259', 'HMDB0000943', 'HMDB0000182', 'HMDB0006344', 'HMDB0000187', 'HMDB0000594', 'HMDB0000755', 'HMDB0000210', 'HMDB0028691', 'HMDB0000131', 'HMDB0000299', 'HMDB0000163', 'HMDB0000252', 'HMDB0000139', 'HMDB0002000', 'HMDB0000695', 'HMDB0001316', 'HMDB0000562', 'HMDB0000289', 'HMDB0000660', 'HMDB0000134', 'HMDB0000826', 'HMDB0000532', 'HMDB0000714', 'HMDB0000125']
pi3k = ['HMDB0000050', 'HMDB0003229', 'HMDB0000214', 'HMDB0001173', 'HMDB0000939', 'HMDB0000696', 'HMDB0000162', 'HMDB0000294', 'HMDB0000806', 'HMDB0000517', 'HMDB0000673', 'HMDB0000133', 'HMDB0000641', 'HMDB0000159', 'HMDB0000848', 'HMDB0000296', 'HMDB0000300', 'HMDB0000157', 'HMDB0000195', 'HMDB0000224', 'HMDB0000904', 'HMDB0000684', 'HMDB0000086', 'HMDB0000177', 'HMDB0000638', 'HMDB0000158', 'HMDB0000827', 'HMDB0000167', 'HMDB0000929', 'HMDB0000254', 'HMDB0000043', 'HMDB0000870', 'HMDB0000064', 'HMDB0000071', 'HMDB0010382', 'HMDB0003045', 'HMDB0000812', 'HMDB0000168', 'HMDB0000759', 'HMDB0002259', 'HMDB0000191', 'HMDB0006344', 'HMDB0000123', 'HMDB0000682', 'HMDB0000187', 'HMDB0000594', 'HMDB0000687', 'HMDB0000156', 'HMDB0000755', 'HMDB0000767', 'HMDB0000163', 'HMDB0000252', 'HMDB0000139', 'HMDB0002000', 'HMDB0000695', 'HMDB0000562', 'HMDB0000289', 'HMDB0000660', 'HMDB0000097', 'HMDB0000134', 'HMDB0000826', 'HMDB0000532', 'HMDB0000714', 'HMDB0011172', 'HMDB0000011', 'HMDB0000194', 'HMDB0000056']

In [ ]:
from matplotlib_venn import venn2

set1 = set(myc)
set2 = set(pi3k)

venn2([set1, set2], set_labels=('MYC', 'PI3K'))
plt.title('Overlap Between Correct MYC And PI3K Case Metabolites')
plt.savefig('Fudan_MYC_PI3K_common_correct_metabolites_venn.svg', bbox_inches = 'tight')
plt.show()

In [ ]:
for i in set1.intersection(set2):
  print(i)